In [21]:
import sympy as sp
from sympy import S, oo, Interval, Union, FiniteSet, EmptySet
from sympy.calculus.util import continuous_domain
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor
)

# I. Vẽ bảng biến thiên của hàm số cho trước

## Hướng dẫn sử dụng chương trình in bảng biến thiên

## 1. Mục đích
Chương trình dùng để:

- Nhập **biểu thức hàm số** \( f(x) \)
- Tự động:
  - tìm **miền xác định**
  - tính **đạo hàm**
  - xác định **các điểm tới hạn**
  - xét dấu đạo hàm
  - in ra **tóm tắt khảo sát**
  - in ra **bảng biến thiên**

Chương trình được viết theo hướng **OOP** và sử dụng thư viện **SymPy** để thao tác với hàm số.

---

## 2. Đầu vào
Đầu vào của chương trình là:

- `expr_input`: chuỗi biểu thức hàm số theo cú pháp Python/SymPy

Ví dụ:

- `"x**3 - 3*x + 1"`
- `"sqrt(x)/(x-1)"`
- `"sin(x) + x**2"`
- `"ln(x)"`
- `"E**x - cos(2*x)"`

> **Lưu ý:**  
> Trong SymPy:
> - lũy thừa nên viết `**`
> - \(\ln(x)\) viết là `log(x)` hoặc `ln(x)` nếu đã khai báo hỗ trợ
> - hằng số \(e\) nên viết là `E` hoặc `exp(1)`
> - căn bậc hai viết là `sqrt(x)`

---

## 3. Mẫu chạy chương trình

### 3.1. Ví dụ 1

```python
# Nhập biểu thức hàm số
expr_input = "x**3 - 3*x + 1"

# Tạo đối tượng khảo sát bảng biến thiên
bbt = BangBienThien(expr_input)

# In phần tóm tắt khảo sát
bbt.print_summary()

# In bảng biến thiên
bbt.print_variation_table()

In [22]:
class BangBienThien:
    """
    Lớp khảo sát và in bảng biến thiên cho hàm số 1 biến thực.

    Ghi chú quan trọng:
    - Chương trình phù hợp nhất với các hàm sơ cấp có số hữu hạn mốc khảo sát trên R.
    - Với các hàm có vô số điểm tới hạn trên toàn R như sin(x), cos(x), ...,
      bảng biến thiên trên toàn R là vô hạn. Khi đó chương trình sẽ thông báo
      cần giới hạn thêm một khoảng khảo sát.
    """

    def __init__(self, expr_str: str, variable: str = "x"):
        """
        input:
            expr_str  : chuỗi biểu thức hàm số, ví dụ: "x^3 - 3*x + 1"
            variable  : tên biến, mặc định là x

        output:
            Khởi tạo đối tượng chứa hàm số, miền xác định và đạo hàm.
        """
        self.variable = variable
        self.x = sp.symbols(variable, real=True)

        # Cho phép viết 2x, x^2, ln(x), sqrt(x), ...
        transformations = standard_transformations + (
            implicit_multiplication_application,
            convert_xor
        )

        local_dict = {
            variable: self.x,
            "ln": sp.log,
            "log": sp.log,
            "sqrt": sp.sqrt,
            "sin": sp.sin,
            "cos": sp.cos,
            "tan": sp.tan,
            "cot": sp.cot,
            "asin": sp.asin,
            "acos": sp.acos,
            "atan": sp.atan,
            "exp": sp.exp,
            "abs": sp.Abs,
            "pi": sp.pi,
            "e": sp.E,
            "E": sp.E
        }

        # Bước 1: phân tích biểu thức
        self.expr = sp.simplify(
            parse_expr(expr_str, transformations=transformations, local_dict=local_dict)
        )

        # Bước 2: miền xác định trên R
        self.domain = continuous_domain(self.expr, self.x, S.Reals)

        # Bước 3: đạo hàm
        self.dexpr = sp.simplify(sp.diff(self.expr, self.x))

        # Kết quả phân tích sẽ được lưu sau khi gọi analyze()
        self.analysis = None

    # =========================================================
    # Các hàm phụ trợ
    # =========================================================
    def _fmt(self, value) -> str:
        """
        Chuyển một đối tượng SymPy sang chuỗi ngắn gọn để in ra bảng.
        """
        value = sp.simplify(value)

        if value == oo:
            return "+∞"
        if value == -oo:
            return "-∞"
        if value == sp.zoo:
            return "zoo"
        if value == sp.nan:
            return "nan"
        return sp.sstr(value)

    def _sort_key(self, value):
        """
        Khóa sắp xếp cho các mốc thực.
        """
        if value == -oo:
            return float("-inf")
        if value == oo:
            return float("inf")
        return float(sp.N(value))

    def _extract_interval_components(self, domain_set):
        """
        Tách miền xác định thành danh sách các khoảng liên thông.
        Chỉ xử lý phần dạng khoảng; các điểm cô lập không đưa vào bảng biến thiên chuẩn.
        """
        if domain_set == S.Reals:
            return [Interval.open(-oo, oo)]

        if isinstance(domain_set, Interval):
            return [domain_set]

        if isinstance(domain_set, Union):
            intervals = []
            for part in domain_set.args:
                if part == S.Reals:
                    intervals.append(Interval.open(-oo, oo))
                elif isinstance(part, Interval):
                    intervals.append(part)
                # Nếu là FiniteSet thì bỏ qua trong bảng biến thiên chuẩn
            intervals.sort(key=lambda I: self._sort_key(I.start))
            return intervals

        return []

    def _extract_finite_points(self, set_obj):
        """
        Tách các điểm hữu hạn, thực từ một tập SymPy.
        Dùng để lấy các điểm mà f'(x) không xác định nhưng f(x) xác định.
        """
        if set_obj in (EmptySet, S.EmptySet):
            return []

        if isinstance(set_obj, FiniteSet):
            points = []
            for p in set_obj:
                if (p.is_real is True) and (p not in (oo, -oo)):
                    points.append(sp.simplify(p))
            return points

        if isinstance(set_obj, Union):
            points = []
            for part in set_obj.args:
                points.extend(self._extract_finite_points(part))
            return points

        # Nếu xuất hiện Interval ở đây thì nghĩa là đạo hàm không xác định trên cả một khoảng,
        # không phù hợp với mẫu bảng biến thiên hữu hạn thông thường.
        if isinstance(set_obj, Interval):
            raise NotImplementedError(
                "Đạo hàm không xác định trên cả một khoảng. "
                "Trường hợp này cần xử lý riêng."
            )

        return []

    def _strict_inside(self, interval: Interval, point) -> bool:
        """
        Kiểm tra điểm có nằm trong nội bộ khoảng hay không.
        """
        if point == interval.start or point == interval.end:
            return False
        return bool(interval.contains(point))

    def _pick_test_point(self, left, right):
        """
        Chọn một điểm thử trong khoảng mở (left, right) để xét dấu f'(x).
        """
        if left == -oo and right == oo:
            return sp.Integer(0)

        if left == -oo:
            return sp.simplify(right - 1)

        if right == oo:
            return sp.simplify(left + 1)

        return sp.simplify((left + right) / 2)

    def _sign_of(self, value) -> str:
        """
        Trả về dấu của một giá trị: '+', '-', '0' hoặc '?'.
        """
        value = sp.simplify(value)

        if value.is_zero is True:
            return "0"
        if value.is_positive is True:
            return "+"
        if value.is_negative is True:
            return "-"

        # Dự phòng: đánh giá số gần đúng
        numeric_value = sp.N(value)
        try:
            numeric_value = float(numeric_value)
            eps = 1e-12
            if numeric_value > eps:
                return "+"
            if numeric_value < -eps:
                return "-"
            return "0"
        except Exception:
            return "?"

    def _sign_on_interval(self, left, right) -> str:
        """
        Xét dấu f'(x) trên khoảng mở (left, right).
        """
        if self.dexpr.equals(0):
            return "0"

        test_point = self._pick_test_point(left, right)
        value = sp.simplify(self.dexpr.subs(self.x, test_point))
        return self._sign_of(value)

    def _boundary_marker(self, point, side: str, closed: bool):
        """
        Tạo thông tin cho đầu mút của một khoảng liên thông trong miền xác định.

        side:
            'left'  -> đầu trái của khoảng
            'right' -> đầu phải của khoảng

        closed:
            True  -> đầu mút thuộc miền xác định
            False -> đầu mút không thuộc miền xác định, cần lấy giới hạn một phía
        """
        if point == -oo:
            return {
                "x_label": "-∞",
                "y_value": sp.limit(self.expr, self.x, -oo),
                "d_label": "",
                "kind": "boundary"
            }

        if point == oo:
            return {
                "x_label": "+∞",
                "y_value": sp.limit(self.expr, self.x, oo),
                "d_label": "",
                "kind": "boundary"
            }

        if closed:
            # Đầu mút thuộc miền xác định => dùng giá trị thật của hàm số tại điểm đó
            y_value = sp.simplify(self.expr.subs(self.x, point))
            x_label = self._fmt(point)
        else:
            # Đầu mút không thuộc miền xác định => dùng giới hạn một phía
            if side == "left":
                y_value = sp.limit(self.expr, self.x, point, dir="+")
                x_label = f"{self._fmt(point)}+"
            else:
                y_value = sp.limit(self.expr, self.x, point, dir="-")
                x_label = f"{self._fmt(point)}-"

        return {
            "x_label": x_label,
            "y_value": y_value,
            "d_label": "",
            "kind": "boundary"
        }

    def _critical_marker(self, point, d_label: str):
        """
        Tạo thông tin cho điểm tới hạn:
        - d_label = '0'   nếu f'(x) = 0
        - d_label = 'KXD' nếu f'(x) không xác định nhưng f(x) xác định
        """
        return {
            "x_label": self._fmt(point),
            "y_value": sp.simplify(self.expr.subs(self.x, point)),
            "d_label": d_label,
            "kind": "critical"
        }

    def _critical_points_in_interval(self, interval: Interval):
        """
        Tìm các điểm tới hạn bên trong một khoảng liên thông của miền xác định.

        Bước giải tích:
        - nghiệm của f'(x) = 0;
        - các điểm mà f'(x) không xác định nhưng f(x) xác định.
        """
        criticals = {}

        # Trường hợp hàm hằng trên khoảng: f'(x) = 0 mọi nơi
        if self.dexpr.equals(0):
            return []

        # -----------------------------------------------------
        # 1) Nghiệm của f'(x) = 0
        # -----------------------------------------------------
        try:
            sol_zero = sp.solveset(sp.Eq(self.dexpr, 0), self.x, domain=interval)
        except Exception as exc:
            raise NotImplementedError(
                "SymPy không giải được phương trình f'(x)=0 một cách chính xác."
            ) from exc

        if sol_zero not in (EmptySet, S.EmptySet):
            if isinstance(sol_zero, FiniteSet):
                for p in sol_zero:
                    if (p.is_real is True) and self._strict_inside(interval, p):
                        criticals[sp.simplify(p)] = "0"
            else:
                raise NotImplementedError(
                    "Đạo hàm có vô số nghiệm hoặc SymPy không biểu diễn được "
                    "hữu hạn các nghiệm của f'(x)=0. "
                    "Hãy giới hạn thêm khoảng khảo sát."
                )

        # -----------------------------------------------------
        # 2) Điểm mà f'(x) không xác định nhưng f(x) xác định
        # -----------------------------------------------------
        try:
            d_domain_on_interval = continuous_domain(self.dexpr, self.x, interval)
            bad_points_set = interval - d_domain_on_interval
            bad_points = self._extract_finite_points(bad_points_set)

            for p in bad_points:
                if self._strict_inside(interval, p):
                    # setdefault để giữ "0" nếu điểm đó đã là nghiệm f'(x)=0
                    criticals.setdefault(sp.simplify(p), "KXD")
        except Exception:
            # Có những biểu thức SymPy không tách được miền xác định của đạo hàm.
            # Khi đó ta bỏ qua nhánh này thay vì làm chương trình dừng.
            pass

        result = [
            {"x": p, "d_label": criticals[p]}
            for p in sorted(criticals.keys(), key=self._sort_key)
        ]
        return result

    def _interval_text(self, left, right) -> str:
        """
        Viết khoảng mở (left, right) dưới dạng chuỗi để mô tả tính đơn điệu.
        """
        left_text = "-∞" if left == -oo else self._fmt(left)
        right_text = "+∞" if right == oo else self._fmt(right)
        return f"({left_text}, {right_text})"

    def _arrow_of(self, sign: str) -> str:
        """
        Đổi dấu của f'(x) sang mũi tên biến thiên của f(x).
        """
        if sign == "+":
            return "↗"
        if sign == "-":
            return "↘"
        if sign == "0":
            return "→"
        return "?"

    # =========================================================
    # Phân tích chính
    # =========================================================
    def analyze(self):
        """
        Phân tích đầy đủ để lập bảng biến thiên.
        """
        components = self._extract_interval_components(self.domain)

        if not components:
            raise ValueError("Không tìm được khoảng liên thông nào trong miền xác định.")

        analyzed_components = []

        for interval in components:
            criticals = self._critical_points_in_interval(interval)

            # Tạo mốc đầu trái và đầu phải của khoảng liên thông
            left_marker = self._boundary_marker(
                interval.start,
                side="left",
                closed=not interval.left_open
            )
            right_marker = self._boundary_marker(
                interval.end,
                side="right",
                closed=not interval.right_open
            )

            # Tạo các điểm tới hạn nội bộ
            critical_markers = [
                self._critical_marker(item["x"], item["d_label"])
                for item in criticals
            ]

            points = [left_marker] + critical_markers + [right_marker]

            # Tạo các khoảng con để xét dấu f'(x)
            split_points = [interval.start] + [item["x"] for item in criticals] + [interval.end]

            subintervals = []
            interval_signs = []

            for i in range(len(split_points) - 1):
                a = split_points[i]
                b = split_points[i + 1]
                subintervals.append((a, b))
                interval_signs.append(self._sign_on_interval(a, b))

            analyzed_components.append({
                "interval": interval,
                "points": points,
                "criticals": criticals,
                "subintervals": subintervals,
                "interval_signs": interval_signs
            })

        self.analysis = {
            "expr": self.expr,
            "domain": self.domain,
            "derivative": self.dexpr,
            "components": analyzed_components
        }
        return self.analysis

    # =========================================================
    # In báo cáo
    # =========================================================
    def print_summary(self):
        """
        In ra:
        - hàm số
        - miền xác định
        - đạo hàm
        - các khoảng đồng biến / nghịch biến / hằng
        - các điểm cực trị nếu suy ra được từ sự đổi dấu của f'(x)
        """
        if self.analysis is None:
            self.analyze()

        print("f(x)  =", self._fmt(self.analysis["expr"]))
        print("D     =", self._fmt(self.analysis["domain"]))
        print("f'(x) =", self._fmt(self.analysis["derivative"]))
        print()

        increasing = []
        decreasing = []
        constant = []
        local_max = []
        local_min = []

        for comp in self.analysis["components"]:
            signs = comp["interval_signs"]
            subs = comp["subintervals"]
            criticals = comp["criticals"]

            for sign, (a, b) in zip(signs, subs):
                text = self._interval_text(a, b)
                if sign == "+":
                    increasing.append(text)
                elif sign == "-":
                    decreasing.append(text)
                elif sign == "0":
                    constant.append(text)

            # Suy cực trị từ đổi dấu của f'(x)
            for i in range(1, len(signs)):
                left_sign = signs[i - 1]
                right_sign = signs[i]
                point = criticals[i - 1]["x"]
                value = sp.simplify(self.expr.subs(self.x, point))

                if left_sign == "+" and right_sign == "-":
                    local_max.append((point, value))
                elif left_sign == "-" and right_sign == "+":
                    local_min.append((point, value))

        print("Khoảng đồng biến :",
              ", ".join(increasing) if increasing else "Không có")
        print("Khoảng nghịch biến:",
              ", ".join(decreasing) if decreasing else "Không có")
        print("Khoảng hằng      :",
              ", ".join(constant) if constant else "Không có")

        if local_max:
            print("Cực đại:")
            for x0, y0 in local_max:
                print(f"  x = {self._fmt(x0)},  f(x) = {self._fmt(y0)}")
        else:
            print("Cực đại: Không có")

        if local_min:
            print("Cực tiểu:")
            for x0, y0 in local_min:
                print(f"  x = {self._fmt(x0)},  f(x) = {self._fmt(y0)}")
        else:
            print("Cực tiểu: Không có")

    def print_variation_table(self):
        """
        In bảng biến thiên gồm 3 dòng:
        - dòng x
        - dòng f'(x)
        - dòng f(x)

        Ý tưởng bố cục:
        - Cột vị trí mốc: ghi giá trị x
        - Cột xen giữa: ghi dấu f'(x) và mũi tên biến thiên
        """
        if self.analysis is None:
            self.analyze()

        x_cells = []
        d_cells = []
        y_cells = []

        components = self.analysis["components"]

        for idx, comp in enumerate(components):
            points = comp["points"]
            signs = comp["interval_signs"]

            # Duyệt theo dạng:
            # điểm - khoảng - điểm - khoảng - ... - điểm
            for i, point_info in enumerate(points):
                x_cells.append(point_info["x_label"])
                d_cells.append(point_info["d_label"])
                y_cells.append(self._fmt(point_info["y_value"]))

                if i < len(signs):
                    x_cells.append("")
                    d_cells.append(signs[i])
                    y_cells.append(self._arrow_of(signs[i]))

            # Nếu còn thành phần miền xác định khác, thêm vạch ngăn
            if idx < len(components) - 1:
                x_cells.append("||")
                d_cells.append("||")
                y_cells.append("||")

        max_len = max(
            max(len(cell) for cell in x_cells) if x_cells else 0,
            max(len(cell) for cell in d_cells) if d_cells else 0,
            max(len(cell) for cell in y_cells) if y_cells else 0,
            6
        )
        cell_width = max(8, min(max_len + 2, 18))

        def sep_line(num_cells):
            return "+" + "-" * 8 + "+" + "+".join(["-" * cell_width] * num_cells) + "+"

        def make_row(title, cells):
            row = f"| {title:<6} |"
            for cell in cells:
                row += f"{cell:^{cell_width}}|"
            return row

        print("\nBẢNG BIẾN THIÊN")
        print(sep_line(len(x_cells)))
        print(make_row("x", x_cells))
        print(sep_line(len(d_cells)))
        print(make_row("f'(x)", d_cells))
        print(sep_line(len(y_cells)))
        print(make_row("f(x)", y_cells))
        print(sep_line(len(y_cells)))

In [23]:
# Nhập biểu thức hàm số
expr_input = "sqrt(x)/(x-1)"

# Tạo đối tượng khảo sát bảng biến thiên
bbt = BangBienThien(expr_input)

# In phần tóm tắt khảo sát
bbt.print_summary()

# In bảng biến thiên
bbt.print_variation_table()

f(x)  = sqrt(x)/(x - 1)
D     = Union(Interval.Ropen(0, 1), Interval.open(1, oo))
f'(x) = (-x - 1)/(2*sqrt(x)*(x - 1)**2)

Khoảng đồng biến : Không có
Khoảng nghịch biến: (0, 1), (1, +∞)
Khoảng hằng      : Không có
Cực đại: Không có
Cực tiểu: Không có

BẢNG BIẾN THIÊN
+--------+--------+--------+--------+--------+--------+--------+--------+
| x      |   0    |        |   1-   |   ||   |   1+   |        |   +∞   |
+--------+--------+--------+--------+--------+--------+--------+--------+
| f'(x)  |        |   -    |        |   ||   |        |   -    |        |
+--------+--------+--------+--------+--------+--------+--------+--------+
| f(x)   |   0    |   ↘    |   -∞   |   ||   |   +∞   |   ↘    |   0    |
+--------+--------+--------+--------+--------+--------+--------+--------+


# II. Kiểm tra khoảng ly nghiệm và xét dấu của \(f'(x)\), \(f''(x)\) trên đoạn \([a,b]\)

## Hướng dẫn sử dụng chương trình kiểm tra khoảng ly nghiệm

## 1. Mục đích
Chương trình dùng để:

- Nhập **biểu thức hàm số** \(f(x)\)
- Nhập đoạn khảo sát \([a,b]\)
- Tự động:
  - tính \(f(a)\), \(f(b)\)
  - kiểm tra điều kiện \(f(a)\cdot f(b) < 0\)
  - tính đạo hàm cấp 1: \(f'(x)\)
  - tính đạo hàm cấp 2: \(f''(x)\)
  - ước lượng giá trị nhỏ nhất và lớn nhất của \(f'(x)\) trên \([a,b]\)
  - ước lượng giá trị nhỏ nhất và lớn nhất của \(f''(x)\) trên \([a,b]\)
  - kết luận \(f'(x)\), \(f''(x)\) có đổi dấu hay không trên \([a,b]\)

Chương trình được viết theo hướng **OOP** và sử dụng thư viện **SymPy** để thao tác với hàm số.

---

## 2. Ý tưởng giải tích
Muốn kiểm tra \((a,b)\) là khoảng ly nghiệm hợp lệ, ta xét:

- \(f(a)\cdot f(b) < 0\)

Nếu điều kiện này đúng thì tiếp tục kiểm tra dấu của \(f'(x)\) và \(f''(x)\) trên \([a,b]\).

### Xét dấu của \(f'(x)\)
Gọi:

- \(m_1 = \min_{x\in[a,b]} f'(x)\)
- \(M_1 = \max_{x\in[a,b]} f'(x)\)

Nếu:
\[
m_1 \cdot M_1 > 0
\]
thì \(f'(x)\) **không đổi dấu** trên \([a,b]\).

- Nếu \(m_1 > 0\) và \(M_1 > 0\) thì \(f'(x) > 0\) trên \([a,b]\)
- Nếu \(m_1 < 0\) và \(M_1 < 0\) thì \(f'(x) < 0\) trên \([a,b]\)

### Xét dấu của \(f''(x)\)
Tương tự, gọi:

- \(m_2 = \min_{x\in[a,b]} f''(x)\)
- \(M_2 = \max_{x\in[a,b]} f''(x)\)

Nếu:
\[
m_2 \cdot M_2 > 0
\]
thì \(f''(x)\) **không đổi dấu** trên \([a,b]\).

---

## 3. Đầu vào
Đầu vào của chương trình là:

- `expr_input`: chuỗi biểu thức hàm số theo cú pháp Python/SymPy
- `a`: đầu trái đoạn khảo sát
- `b`: đầu phải đoạn khảo sát

Ví dụ:

- `"x**3 - x - 1"`
- `"E**x - cos(2*x)"`
- `"x**5 - 3*x**3 + 2*x**2 - x + 5"`

> **Lưu ý:**  
> Trong SymPy:
> - lũy thừa viết là `**`
> - \(\ln(x)\) viết là `log(x)` hoặc `ln(x)`
> - hằng số \(e\) viết là `E`
> - \(\pi\) viết là `pi`

---

In [24]:
class KiemTraKhoangLyNghiem:
    """
    Lớp kiểm tra khoảng ly nghiệm (a, b) và xét dấu của f'(x), f''(x) trên [a, b].

    Ý tưởng:
    - Bước 1: kiểm tra f(a) * f(b) < 0
    - Bước 2: nếu hợp lệ thì tính f'(x), f''(x)
    - Bước 3: tìm min, max của f'(x), f''(x) trên [a, b]
    - Bước 4: xét dấu dựa vào tích min * max
    """

    def __init__(self, expr_str: str, a, b, variable: str = "x"):
        """
        input:
            expr_str : chuỗi biểu thức hàm số
            a, b     : hai đầu mút đoạn khảo sát
            variable : tên biến, mặc định là x

        output:
            Khởi tạo đối tượng kiểm tra khoảng ly nghiệm.
        """
        self.variable = variable
        self.x = sp.symbols(variable, real=True)

        transformations = standard_transformations + (
            implicit_multiplication_application,
            convert_xor
        )

        local_dict = {
            variable: self.x,
            "ln": sp.log,
            "log": sp.log,
            "sqrt": sp.sqrt,
            "sin": sp.sin,
            "cos": sp.cos,
            "tan": sp.tan,
            "cot": sp.cot,
            "asin": sp.asin,
            "acos": sp.acos,
            "atan": sp.atan,
            "exp": sp.exp,
            "abs": sp.Abs,
            "pi": sp.pi,
            "e": sp.E,
            "E": sp.E
        }

        self.expr = sp.simplify(
            parse_expr(expr_str, transformations=transformations, local_dict=local_dict)
        )

        self.a = sp.nsimplify(a)
        self.b = sp.nsimplify(b)

        if self.a >= self.b:
            raise ValueError("Cần có a < b.")

        self.d1 = sp.simplify(sp.diff(self.expr, self.x))
        self.d2 = sp.simplify(sp.diff(self.d1, self.x))

    # =========================================================
    # Hàm phụ trợ
    # =========================================================
    def _fmt(self, value) -> str:
        """
        Đưa biểu thức về dạng chuỗi gọn để in ra màn hình.
        """
        value = sp.simplify(value)
        return sp.sstr(value)

    def _float(self, value) -> float:
        """
        Đổi một giá trị SymPy sang float.
        """
        return float(sp.N(value, 15))

    def _safe_eval(self, func_expr, point):
        """
        Tính giá trị gần đúng của biểu thức tại một điểm.
        """
        return sp.N(func_expr.subs(self.x, point), 15)

    def _find_stationary_points_numeric(self, func_expr, samples=40):
        """
        Tìm gần đúng các nghiệm của func_expr'(x) = 0 trên [a, b]
        bằng cách quét nhiều điểm đầu và dùng nsolve.

        Mục đích:
        - Dùng để tìm các điểm ứng viên khi xét min/max của func_expr trên [a, b].
        """
        derivative = sp.simplify(sp.diff(func_expr, self.x))
        roots = []

        a_float = self._float(self.a)
        b_float = self._float(self.b)

        if samples < 2:
            samples = 2

        grid = [a_float + (b_float - a_float) * i / samples for i in range(samples + 1)]

        for guess in grid:
            try:
                root = sp.nsolve(derivative, self.x, guess, tol=1e-14, maxsteps=100)
                root = sp.N(root, 15)

                # chỉ giữ nghiệm thực trong [a,b]
                if abs(sp.im(root)) < 1e-10:
                    root_real = float(sp.re(root))
                    if a_float - 1e-10 <= root_real <= b_float + 1e-10:
                        # loại nghiệm trùng
                        is_new = True
                        for r in roots:
                            if abs(root_real - r) < 1e-7:
                                is_new = False
                                break
                        if is_new:
                            roots.append(root_real)
            except Exception:
                pass

        roots.sort()
        return roots

    def _find_min_max_on_segment(self, func_expr):
        """
        Tìm gần đúng min và max của func_expr trên [a, b].

        Ý tưởng:
        - lấy các điểm ứng viên gồm:
          + a, b
          + các nghiệm của func_expr'(x)=0 trên [a,b]
        - thay vào để so sánh giá trị.
        """
        candidates = [self._float(self.a), self._float(self.b)]
        candidates += self._find_stationary_points_numeric(func_expr)

        values = []
        for t in candidates:
            try:
                val = float(sp.N(func_expr.subs(self.x, t), 15))
                values.append((t, val))
            except Exception:
                pass

        if not values:
            raise ValueError("Không tính được min/max trên đoạn đã cho.")

        min_point, min_value = min(values, key=lambda item: item[1])
        max_point, max_value = max(values, key=lambda item: item[1])

        return {
            "min_point": min_point,
            "min_value": min_value,
            "max_point": max_point,
            "max_value": max_value,
            "candidates": values
        }

    def _sign_conclusion(self, min_value, max_value, eps=1e-12):
        """
        Kết luận dấu của một hàm trên [a,b] từ min và max.
        """
        if min_value * max_value > 0:
            if min_value > 0 and max_value > 0:
                return "Luôn dương trên [a, b]"
            if min_value < 0 and max_value < 0:
                return "Luôn âm trên [a, b]"
        elif abs(min_value) <= eps and abs(max_value) <= eps:
            return "Bằng 0 trên [a, b]"
        elif abs(min_value) <= eps or abs(max_value) <= eps:
            return "Không đổi dấu hoàn toàn, có thể chạm 0 trên [a, b]"
        return "Đổi dấu trên [a, b]"

    # =========================================================
    # Phân tích chính
    # =========================================================
    def analyze(self):
        """
        Phân tích khoảng ly nghiệm và dấu của f', f''.
        """
        fa = self._safe_eval(self.expr, self.a)
        fb = self._safe_eval(self.expr, self.b)

        product = sp.N(fa * fb, 15)
        valid_isolation = product < 0

        result = {
            "f": self.expr,
            "a": self.a,
            "b": self.b,
            "fa": fa,
            "fb": fb,
            "fa_fb": product,
            "is_valid_isolation": bool(valid_isolation),
            "f1": self.d1,
            "f2": self.d2
        }

        if valid_isolation:
            d1_info = self._find_min_max_on_segment(self.d1)
            d2_info = self._find_min_max_on_segment(self.d2)

            result["d1_info"] = d1_info
            result["d2_info"] = d2_info
            result["d1_sign"] = self._sign_conclusion(
                d1_info["min_value"], d1_info["max_value"]
            )
            result["d2_sign"] = self._sign_conclusion(
                d2_info["min_value"], d2_info["max_value"]
            )

        return result

    # =========================================================
    # In kết quả
    # =========================================================
    def print_summary(self):
        """
        In báo cáo kiểm tra khoảng ly nghiệm và dấu của f'(x), f''(x).
        """
        data = self.analyze()

        print("f(x)    =", self._fmt(data["f"]))
        print("Đoạn xét = [{}, {}]".format(self._fmt(data["a"]), self._fmt(data["b"])))
        print("f(a)    =", data["fa"])
        print("f(b)    =", data["fb"])
        print("f(a)f(b)=", data["fa_fb"])
        print()

        if not data["is_valid_isolation"]:
            print("KẾT LUẬN:")
            print("- (a, b) KHÔNG phải khoảng ly nghiệm hợp lệ vì f(a) * f(b) >= 0.")
            return

        print("KẾT LUẬN:")
        print("- (a, b) là khoảng ly nghiệm hợp lệ vì f(a) * f(b) < 0.")
        print()

        print("Đạo hàm cấp 1:")
        print("f'(x) =", self._fmt(data["f1"]))
        print("min f'(x) =", data["d1_info"]["min_value"],
              "tại x =", data["d1_info"]["min_point"])
        print("max f'(x) =", data["d1_info"]["max_value"],
              "tại x =", data["d1_info"]["max_point"])
        print("Dấu của f'(x):", data["d1_sign"])
        print()

        print("Đạo hàm cấp 2:")
        print("f''(x) =", self._fmt(data["f2"]))
        print("min f''(x) =", data["d2_info"]["min_value"],
              "tại x =", data["d2_info"]["min_point"])
        print("max f''(x) =", data["d2_info"]["max_value"],
              "tại x =", data["d2_info"]["max_point"])
        print("Dấu của f''(x):", data["d2_sign"])

In [26]:
# Nhập biểu thức hàm số và khoảng ly nghiệm
expr_input = "E**x - cos(2*x)"
a = -1
b = -0.01

# Tạo đối tượng kiểm tra
kt = KiemTraKhoangLyNghiem(expr_input, a, b)

# In kết quả kiểm tra
kt.print_summary()

f(x)    = exp(x) - cos(2*x)
Đoạn xét = [-1, -1/100]
f(a)    = 0.784026277718585
f(b)    = -0.00975017291740972
f(a)f(b)= -0.00764439177954930

KẾT LUẬN:
- (a, b) là khoảng ly nghiệm hợp lệ vì f(a) * f(b) < 0.

Đạo hàm cấp 1:
f'(x) = exp(x) + 2*sin(2*x)
min f'(x) = -1.5563769512739998 tại x = -0.8394947304223329
max f'(x) = 0.950052500362502 tại x = -0.01
Dấu của f'(x): Đổi dấu trên [a, b]

Đạo hàm cấp 2:
f''(x) = exp(x) + 4*cos(2*x)
min f''(x) = -1.2967079050171273 tại x = -1.0
max f''(x) = 4.989249860415479 tại x = -0.01
Dấu của f''(x): Đổi dấu trên [a, b]
